In [1]:
import sys, os, subprocess
from pathlib import Path

# Colab: clone repo and install deps. Local: resolve root from CWD.
try:
    import google.colab  # noqa
    REPO = '/content/Katabatic'
    if not os.path.exists(REPO):
        subprocess.run(
            ['git', 'clone', 'https://github.com/lukebrumby/katabatic-personal.git', REPO],
            check=True
        )
    os.chdir(REPO)
    sys.path.insert(0, REPO)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
    ROOT = Path(REPO)
except ImportError:
    ROOT = Path.cwd().resolve()
    for _ in range(5):
        if (ROOT / 'pyproject.toml').exists() or (ROOT / 'raw_data').exists():
            break
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models_luke.ctabganplus.models import CTABGANModel

ROOT: /content/Katabatic


In [ ]:
MODEL = lambda: CTABGANModel(
    epochs=150,
    batch_size=500,
    random_dim=100,
    num_channels=64,
    class_dim=(256, 256, 256, 256),
    l2scale=1e-5,
    seed=42,
)

In [3]:
DATASETS = ["car", "adult", "magic", "shuttle", "nursery"]

for dataset in DATASETS:
    dataset_path = ROOT / "raw_data" / f"{dataset}.csv"
    output_path = ROOT / "discretized_data" / f"{dataset}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Preprocessing {dataset}...")
    discretize_preprocess(str(dataset_path), str(output_path))

Preprocessing car...
Preprocessing: /content/Katabatic/raw_data/car.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/car.csv
Preprocessing adult...
Preprocessing: /content/Katabatic/raw_data/adult.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/adult.csv
Preprocessing magic...
Preprocessing: /content/Katabatic/raw_data/magic.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/magic.csv
Preprocessing shuttle...
Preprocessing: /content/Katabatic/raw_data/shuttle.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/shuttle.csv
Preprocessing nursery...
Preprocessing: /content/Katabatic/raw_data/nursery.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/nursery.csv


In [ ]:
for dataset in DATASETS:
    print(f"\n{'='*60}")
    print(f"CTAB-GAN+ -> {dataset}")
    input_csv = str(ROOT / "discretized_data" / f"{dataset}.csv")
    output_dir = str(ROOT / "sample_data" / dataset)
    synthetic_dir = str(ROOT / "synthetic" / dataset / "ctabganplus")

    pipeline = TrainTestSplitPipeline(model=MODEL)
    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=output_dir,
    )
    print(result)


CTAB-GAN+ -> car
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
[CTAB-GAN+] Preparing data...
[CTAB-GAN+] Initializing synthesizer...
[CTAB-GAN+] Training GAN (epochs=150)...


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base

  [CTAB-GAN+] Epoch 10/150
  [CTAB-GAN+] Epoch 20/150
  [CTAB-GAN+] Epoch 30/150
  [CTAB-GAN+] Epoch 40/150
  [CTAB-GAN+] Epoch 50/150
  [CTAB-GAN+] Epoch 60/150
  [CTAB-GAN+] Epoch 70/150
  [CTAB-GAN+] Epoch 80/150
  [CTAB-GAN+] Epoch 90/150
  [CTAB-GAN+] Epoch 100/150
  [CTAB-GAN+] Epoch 110/150
  [CTAB-GAN+] Epoch 120/150
  [CTAB-GAN+] Epoch 130/150
  [CTAB-GAN+] Epoch 140/150
  [CTAB-GAN+] Epoch 150/150
[CTAB-GAN+] Generating synthetic data...
[CTAB-GAN+] Training complete!
  X -> /content/Katabatic/synthetic/car/ctabganplus/x_synth.csv
  y -> /content/Katabatic/synthetic/car/ctabganplus/y_synth.csv


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [00:37:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/car/ctabganplus_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7110
F1 Score: 0.6576

MLP:
Accuracy: 0.7948
F1 Score: 0.7680

RF:
Accuracy: 0.7428
F1 Score: 0.7201

XGBoost:
Accuracy: 0.7254
F1 Score: 0.7118
Train test split pipeline executed successfully.

CTAB-GAN+ -> adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[CTAB-GAN+] Preparing data...
[CTAB-GAN+] Initializing synthesizer...
[CTAB-GAN+] Training GAN (epochs=150)...


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (9) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/mixture/_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (7) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (6) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklea

  [CTAB-GAN+] Epoch 10/150
  [CTAB-GAN+] Epoch 20/150
  [CTAB-GAN+] Epoch 30/150
  [CTAB-GAN+] Epoch 40/150
  [CTAB-GAN+] Epoch 50/150
  [CTAB-GAN+] Epoch 60/150
  [CTAB-GAN+] Epoch 70/150
  [CTAB-GAN+] Epoch 80/150
  [CTAB-GAN+] Epoch 90/150
  [CTAB-GAN+] Epoch 100/150
  [CTAB-GAN+] Epoch 110/150
